In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# -----------------------------
# File paths
# -----------------------------
DATA_DIR = Path("/Users/evelyn/Desktop/M2/ML/data/raw")

TRAIN_PATH = DATA_DIR / "KDDTrain+.txt"
TEST_PATH = DATA_DIR / "KDDTest+.txt"
FEATURES_PATH = DATA_DIR / "feature_names.txt"

In [2]:
# -----------------------------
# Read feature names
# -----------------------------
def load_feature_names(feature_file: str | Path) -> list[str]:
    """
    Extract the 41 NSL-KDD feature names from feature_names.txt.
    """
    feature_names = []

    with open(feature_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Skip empty lines and comments/header lines
            if not line or line.startswith("|"):
                continue

            # Feature lines look like: duration: continuous.
            if ":" in line:
                feature_name = line.split(":")[0].strip()

                # Ignore the first line listing attack names
                if feature_name not in ["back, buffer_overflow, ftp_write, guess_passwd, imap, ipsweep, land, loadmodule, multihop, neptune, nmap, normal, perl, phf, pod, portsweep, rootkit, satan, smurf, spy, teardrop, warezclient, warezmaster"]:
                    # Only keep actual column names
                    if " " not in feature_name and "," not in feature_name:
                        feature_names.append(feature_name)

    return feature_names


feature_names = load_feature_names(FEATURES_PATH)
columns = feature_names + ["label", "difficulty"]

print(f"Number of input features: {len(feature_names)}")
print("First 5 features:", feature_names[:5])


# -----------------------------
# Load train and test data
# -----------------------------
train_df = pd.read_csv(TRAIN_PATH, header=None, names=columns)
test_df = pd.read_csv(TEST_PATH, header=None, names=columns)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# -----------------------------
# Quick inspection
# -----------------------------
print("\nTrain sample:")
print(train_df.head())

print("\nTrain label distribution:")
print(train_df["label"].value_counts().head(10))

Number of input features: 41
First 5 features: ['duration', 'protocol_type', 'service', 'flag', 'src_bytes']
Train shape: (125973, 43)
Test shape: (22544, 43)

Train sample:
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_di

In [3]:
# -----------------------------
# Create binary target
# normal = 0, attack = 1
# -----------------------------
train_df["target"] = np.where(train_df["label"] == "normal", 0, 1)
test_df["target"] = np.where(test_df["label"] == "normal", 0, 1)

print("\nBinary target distribution (train):")
print(train_df["target"].value_counts())

print("\nBinary target distribution (test):")
print(test_df["target"].value_counts())

# -----------------------------
# Separate features and target
# -----------------------------
X_train = train_df[feature_names].copy()
y_train = train_df["target"].copy()

X_test = test_df[feature_names].copy()
y_test = test_df["target"].copy()

# -----------------------------
# Identify categorical and numerical columns
# -----------------------------
categorical_cols = ["protocol_type", "service", "flag"]
numerical_cols = [col for col in feature_names if col not in categorical_cols]

print("\nCategorical columns:", categorical_cols)
print("Number of numerical columns:", len(numerical_cols))


Binary target distribution (train):
target
0    67343
1    58630
Name: count, dtype: int64

Binary target distribution (test):
target
1    12833
0     9711
Name: count, dtype: int64

Categorical columns: ['protocol_type', 'service', 'flag']
Number of numerical columns: 38


In [4]:
# -----------------------------
# One-hot encode categorical features
# -----------------------------
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

encoded_cat_names = encoder.get_feature_names_out(categorical_cols)

X_train_cat_df = pd.DataFrame(
    X_train_cat,
    columns=encoded_cat_names,
    index=X_train.index
)

X_test_cat_df = pd.DataFrame(
    X_test_cat,
    columns=encoded_cat_names,
    index=X_test.index
)


# -----------------------------
# Scale numerical features
# -----------------------------
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_test_num = scaler.transform(X_test[numerical_cols])

X_train_num_df = pd.DataFrame(
    X_train_num,
    columns=numerical_cols,
    index=X_train.index
)

X_test_num_df = pd.DataFrame(
    X_test_num,
    columns=numerical_cols,
    index=X_test.index
)


In [ ]:
# -----------------------------
# Combine processed features
# -----------------------------
X_train_processed = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_processed = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

print("\nProcessed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

# -----------------------------
# Create normal-only training set
# For autoencoder / PCA anomaly detection
# -----------------------------
X_train_normal = X_train_processed[y_train == 0].copy()
y_train_normal = y_train[y_train == 0].copy()

print("\nNormal-only training shape:", X_train_normal.shape)

# -----------------------------
# Save processed files
# -----------------------------
processed_dir = Path("/Users/evelyn/Desktop/M2/ML/data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

X_train_processed.to_csv(processed_dir / "X_train_processed.csv", index=False)
X_test_processed.to_csv(processed_dir / "X_test_processed.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)
X_train_normal.to_csv(processed_dir / "X_train_normal.csv", index=False)

print("\nSaved processed files to:", processed_dir)


Processed train shape: (125973, 122)
Processed test shape: (22544, 122)

Normal-only training shape: (67343, 122)
